In [29]:
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
import numpy as np

In [30]:
profiles = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client profiles.xlsx")
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR
0,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,More than 200,Finance
1,G1-002,Govt,Yes,NaN,,2020-05-19 00:00:00,More than 200,Security
2,G1-003,Govt,Yes,NaN,,2021-07-18 00:00:00,> 200,Transportation
3,G1-005,Govt,Y,NaN,NaN,2022-02-11 00:00:00,> 200,Security
4,G1-006,Govt,Y,NaN,NaN,2022-03-18 00:00:00,50 ~ 200,Legal


Things to clean:\
    - columns using different values to represent the same info(Sg-based,staff strength)\
    - getting rid of missing values (Country of overseas client)

In [31]:
profiles[['COUNTRY OF OVERSEAS CLIENT','OVERSEAS','SG-BASED']].value_counts()

COUNTRY OF OVERSEAS CLIENT  OVERSEAS  SG-BASED
Malaysia                    Yes                   6
                            Y                     3
India                       Yes                   1
China                       Y                     1
Indonesia                   Yes                   1
Malaysian                   Y                     1
Name: count, dtype: int64

In [32]:
# ---- Clean Client Profiles ----
# Strip stray leading/trailing whitespace on text columns
obj_cols = ['SG-BASED', 'OVERSEAS', 'COUNTRY OF OVERSEAS CLIENT',
            'STAFF STRENGTH', 'SECTOR', 'TYPE', 'CLIENT ID']
for c in obj_cols:
    profiles[c] = profiles[c].apply(lambda v: v.strip() if isinstance(v, str) else v)

# SG-BASED / OVERSEAS: standardise to explicit Y / N ('Yes' and 'Y' -> 'Y', blanks -> 'N')
def yes_no(v):
    return 'Y' if str(v).strip().upper() in ('Y', 'YES') else 'N'
profiles['SG-BASED'] = profiles['SG-BASED'].apply(yes_no)
profiles['OVERSEAS'] = profiles['OVERSEAS'].apply(yes_no)

# COUNTRY OF OVERSEAS CLIENT: unify blanks to NaN, fix 'Malaysian' -> 'Malaysia'
profiles['COUNTRY OF OVERSEAS CLIENT'] = (
    profiles['COUNTRY OF OVERSEAS CLIENT']
    .replace(r'^\s*$', np.nan, regex=True)       # empty / whitespace-only -> NaN
    .replace('Malaysian', 'Malaysia')
)

# Derived COUNTRY: domestic clients -> 'Singapore', otherwise the overseas country
profiles['COUNTRY'] = profiles['COUNTRY OF OVERSEAS CLIENT'].fillna('Singapore')

# STAFF STRENGTH: normalise synonym labels, then derive an ORG SIZE bucket.
# Metadata rule: 1~49 = Small, 50~200 = Medium, >200 = Large.
# 'Less than 200' is ambiguous (spans Small & Medium) -> kept as 'Unspecified'.
profiles['STAFF STRENGTH'] = profiles['STAFF STRENGTH'].replace(
    {'More than 200': '> 200', 'Less than 49': '1 ~ 49'})
size_map = {'1 ~ 49': 'Small', '50 ~ 200': 'Medium',
            '> 200': 'Large', 'Less than 200': 'Unspecified'}
profiles['ORG SIZE'] = profiles['STAFF STRENGTH'].map(size_map)
assert profiles['ORG SIZE'].notna().all(), \
    profiles.loc[profiles['ORG SIZE'].isna(), 'STAFF STRENGTH'].unique()

# COMMENCEMENT DATE: mix of datetime & string -> uniform datetime64
profiles['COMMENCEMENT DATE'] = pd.to_datetime(profiles['COMMENCEMENT DATE'], errors='coerce')
assert profiles['COMMENCEMENT DATE'].notna().all(), "Unparseable COMMENCEMENT DATE remains"

profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,ORG SIZE
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [33]:
profiles['STAFF STRENGTH'].value_counts()

STAFF STRENGTH
> 200            62
50 ~ 200         57
1 ~ 49           22
Less than 200     7
Name: count, dtype: int64

In [34]:
profiles['SG-BASED'].value_counts()

SG-BASED
Y    114
N     34
Name: count, dtype: int64

In [35]:
profiles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CLIENT ID                   148 non-null    object        
 1   TYPE                        148 non-null    object        
 2   SG-BASED                    148 non-null    object        
 3   OVERSEAS                    148 non-null    object        
 4   COUNTRY OF OVERSEAS CLIENT  34 non-null     object        
 5   COMMENCEMENT DATE           148 non-null    datetime64[ns]
 6   STAFF STRENGTH              148 non-null    object        
 7   SECTOR                      148 non-null    object        
 8   COUNTRY                     148 non-null    object        
 9   ORG SIZE                    148 non-null    object        
dtypes: datetime64[ns](1), object(9)
memory usage: 11.7+ KB


In [36]:
survey = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client survey.xlsx")
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING
0,2021,G1-001,3,4,4,5,8
1,2021,G1-002,3,4,4,2,7
2,2021,G1-003,5,4,3,5,8
3,2021,G1-001,3,4,4,5,8
4,2021,G1-002,3,4,4,2,7


In [37]:
survey['CLIENT ID'].value_counts()

CLIENT ID
G1-001    9
G1-002    9
G1-003    9
G1-006    7
N1-001    5
         ..
P3-027    1
P3-028    1
P3-029    1
P3-030    1
P3-031    1
Name: count, Length: 143, dtype: int64

In [38]:
survey.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432 entries, 0 to 431
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   YEAR                      432 non-null    int64 
 1   CLIENT ID                 432 non-null    object
 2   PRESALES AND PARTNERSHIP  432 non-null    int64 
 3   TECHNICAL EXPERTISE       432 non-null    int64 
 4   PROJECT DELIVERY          432 non-null    int64 
 5   POST-SALES SUPPORT        432 non-null    int64 
 6   NPS RATING                432 non-null    int64 
dtypes: int64(6), object(1)
memory usage: 23.8+ KB


In [39]:
survey.isnull().sum()

YEAR                        0
CLIENT ID                   0
PRESALES AND PARTNERSHIP    0
TECHNICAL EXPERTISE         0
PROJECT DELIVERY            0
POST-SALES SUPPORT          0
NPS RATING                  0
dtype: int64

In [40]:
# ---- Clean Client Survey ----
# Drop 30 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
survey = survey.drop_duplicates()
survey = survey[survey['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived NPS category per respondent (Meta Data thresholds)
def nps_category(score):
    if score >= 9:
        return 'Promoter'    # 9-10
    elif score >= 7:
        return 'Passive'     # 7-8
    return 'Detractor'       # 6 or less
survey['NPS CATEGORY'] = survey['NPS RATING'].apply(nps_category)

survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,NPS CATEGORY
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [41]:
RC = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Revenue and COGS.xlsx")
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,2021,148180,17321,25562,41201
1,G1-001,2022,197313,23604,32875,50085
2,G1-001,2023,349899,51332,71695,89089
3,G1-001,2024,268821,40969,56452,76545
4,G1-001,2025,138115,14516,20889,28968


In [42]:
# ---- Clean Revenue and COGS ----
# Drop 21 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
# (P1-010, P1-018, P1-035 -> metadata says such records can be ignored)
RC = RC.drop_duplicates()
RC = RC[RC['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived cost / profitability metrics (formulas from Meta Data)
RC['COGS'] = RC['HARDWARE'] + RC['SOFTWARE'] + RC['MANPOWER']
RC['GROSS PROFIT'] = RC['REVENUE'] - RC['COGS']
RC['GROSS MARGIN %'] = (RC['GROSS PROFIT'] / RC['REVENUE'] * 100).round(2)
RC['GROSS MARKUP %'] = (RC['GROSS PROFIT'] / RC['COGS'] * 100).round(2)
RC['LOSS'] = RC['GROSS PROFIT'] < 0   # negative gross profit (all NPO clients) -> flagged, kept

RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,COGS,GROSS PROFIT,GROSS MARGIN %,GROSS MARKUP %,LOSS
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [43]:
RC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CLIENT ID       392 non-null    object 
 1   YEAR            392 non-null    int64  
 2   REVENUE         392 non-null    int64  
 3   HARDWARE        392 non-null    int64  
 4   SOFTWARE        392 non-null    int64  
 5   MANPOWER        392 non-null    int64  
 6   COGS            392 non-null    int64  
 7   GROSS PROFIT    392 non-null    int64  
 8   GROSS MARGIN %  392 non-null    float64
 9   GROSS MARKUP %  392 non-null    float64
 10  LOSS            392 non-null    bool   
dtypes: bool(1), float64(2), int64(7), object(1)
memory usage: 31.1+ KB


In [44]:
RC.isnull().sum()

CLIENT ID         0
YEAR              0
REVENUE           0
HARDWARE          0
SOFTWARE          0
MANPOWER          0
COGS              0
GROSS PROFIT      0
GROSS MARGIN %    0
GROSS MARKUP %    0
LOSS              0
dtype: int64

In [45]:
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,ORG SIZE
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [46]:
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,NPS CATEGORY
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [47]:
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,COGS,GROSS PROFIT,GROSS MARGIN %,GROSS MARKUP %,LOSS
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [48]:
# ---- Export cleaned datasets ----
import os
os.makedirs('cleaned', exist_ok=True)
profiles.to_csv('cleaned/client_profiles_clean.csv', index=False)
RC.to_csv('cleaned/revenue_cogs_clean.csv', index=False)
survey.to_csv('cleaned/client_survey_clean.csv', index=False)
print('Exported cleaned files:', os.listdir('cleaned'))

Exported cleaned files: ['client_profiles_clean.csv', 'client_survey_clean.csv', 'revenue_cogs_clean.csv']


## DB 1 (Elliot)

#### Graph 1

In [49]:
df = (RC
      .merge(profiles, on="CLIENT ID", how="left")
      .merge(survey,   on=["CLIENT ID", "YEAR"], how="inner"))

In [54]:
CATS = ["Detractor", "Passive", "Promoter"]                       # legend / dropdown order
COLOURS = {"Detractor": "#d62728", "Passive": "#9aa0a6", "Promoter": "#2ca02c"}
TREND = "Trend for this year"

# One least-squares line per YEAR, in plain pandas:
#   slope = cov(x, y) / var(x),  intercept = mean(y) - slope * mean(x)
# Two endpoints are enough to draw a straight line.
def yearly_trend(g):
    slope = g["NPS RATING"].cov(g["GROSS MARGIN %"]) / g["NPS RATING"].var()
    intercept = g["GROSS MARGIN %"].mean() - slope * g["NPS RATING"].mean()
    x = pd.Series([4, 10])                                        # span the full NPS scale
    return pd.DataFrame({"NPS RATING": x, "GROSS MARGIN %": intercept + slope * x})

trend = (df.groupby("YEAR")[["NPS RATING", "GROSS MARGIN %"]]
           .apply(yearly_trend)
           .reset_index())
trend["SERIES"] = TREND
trend

,YEAR,level_1,NPS RATING,GROSS MARGIN %,SERIES
0,2021,0,4,43.593982,Trend for this year
1,2021,1,10,42.722741,Trend for this year
2,2022,0,4,48.722021,Trend for this year
3,2022,1,10,42.916064,Trend for this year
4,2023,0,4,44.710546,Trend for this year
5,2023,1,10,38.738449,Trend for this year
6,2024,0,4,46.965845,Trend for this year
7,2024,1,10,39.626624,Trend for this year
8,2025,0,4,50.496931,Trend for this year
9,2025,1,10,39.446947,Trend for this year


In [57]:
fig = px.scatter(
    df,
    x="NPS RATING",
    y="GROSS MARGIN %",
    size="REVENUE",                       # bubble size = deal size
    color="NPS CATEGORY",
    color_discrete_map=COLOURS,           # red = detractor, grey = passive, green = promoter
    category_orders={"NPS CATEGORY": CATS},
    animation_frame="YEAR",               # -> year slider + play button
    animation_group="CLIENT ID",          # same client morphs smoothly between years
    hover_name="CLIENT ID",
    hover_data={"SECTOR": True, "TYPE": True, "ORG SIZE": True,
                "REVENUE": ":,.0f", "GROSS MARGIN %": ":.1f"},
    size_max=45,
    range_x=[3.5, 10.5],                 # fixed axes so the animation doesn't jump
    range_y=[-40, 70],
    title="Do happier client earn bring in more profit?",
)

# Animated trendline: px.line gives us one line trace per year, which we splice into
# the scatter's frames so the fit is redrawn every time the year slider moves.
trend_fig = px.line(trend, x="NPS RATING", y="GROSS MARGIN %",
                    animation_frame="YEAR", color="SERIES",
                    color_discrete_sequence=["#333"])

fig.add_traces(trend_fig.data)                     # line for the year the slider starts on
lines_by_year = {f.name: f.data for f in trend_fig.frames}
for frame in fig.frames:                           # ...and for every other year
    frame.data = frame.data + lines_by_year[frame.name]

# Reference lines
overall_margin = df["GROSS PROFIT"].sum() / df["REVENUE"].sum() * 100
fig.add_hline(y=overall_margin, line_dash="dot", line_color="#555",
              annotation_text=f"Overall margin {overall_margin:.1f}%",
              annotation_position="top left")
fig.add_hline(y=0, line_color="#d62728", line_width=1,
              annotation_text="Break-even", annotation_position="bottom right")

# 2nd interactive element: dropdown to isolate one NPS category (the trendline stays on).
# Append to px's updatemenus -- replacing the list would delete the Play/Pause buttons.
buttons = [dict(label="All categories", method="update",
                args=[{"visible": [True] * len(fig.data)}])]
buttons += [dict(label=f"{cat} only", method="update",
                 args=[{"visible": [t.name in (cat, TREND) for t in fig.data]}])
            for cat in CATS]

fig.update_layout(
    updatemenus=list(fig.layout.updatemenus) + [
        dict(buttons=buttons, x=1.0, xanchor="right", y=1.12, yanchor="top")],
    xaxis_title="NPS Rating (0-10)",
    yaxis_title="Gross Margin (%)",
    template="plotly_white",
    height=600,
)

fig.show()

### Insights

1. Happier clients = thinner margins. Gross margin falls from 46.6% (Detractors) → 43.4% (Passives) → 40.6% (Promoters), even though Promoters bring the highest revenue (~$390k vs ~$279k). Holds even after excluding NPO clients.
2. The gap is widening every year. Trend line was flat in 2021 (−0.15% margin per NPS point) but steepens to −1.85% by 2025 — top-rated clients now earn us ~10pts less margin than low-rated ones.
3. All 5 loss-making engagements are high-NPS charity clients. They rate us 9+ while we lose money serving them (e.g. N1-002, N1-007 for two years running) — discounting is unmanaged, not strategic.

#### Graph 2

#### Graph 3

## DB 2 (Zafran)

#### Graph 1

In [52]:
fig = px.sunburst(profiles,
    path=["COUNTRY","TYPE","SECTOR",],
    # values='STAFF STRENGTH',
    title='sunburst plot showing the country, type and sector',
    height = 700,
    width = 700
)
fig.show()

#### Graph 2

In [53]:
merged_df1 = pd.merge(profiles, RC, on='CLIENT ID', how='inner') #used AI for this in Scatterplot of year against country mean profit chat
ProfitPerCountry = merged_df1.groupby(['COUNTRY', 'YEAR'])['GROSS PROFIT'].sum().reset_index()

fig = go.Figure()

for country in ProfitPerCountry['COUNTRY'].unique():
    subset = ProfitPerCountry[ProfitPerCountry['COUNTRY'] == country]
    fig.add_trace(
        go.Scatter(
            x=subset['YEAR'],
            y=subset['GROSS PROFIT'],
            name=country,
            mode='lines+markers'
        )
    )

fig.update_layout(
    title='Mean Profit Per Country Timeline',
    xaxis_title='Year',
    yaxis_title='Mean Profit',
    height=1000
)

fig.show()

#### Graph 3

## DB 3 (name)

#### Graph 1

#### Graph 2

#### Graph 3